# LGU-RAG v2: PDF RAG Assistant with Gemini + ChromaDB + Query Cache

**What this notebook does (matches the architecture diagram):**
1. Upload PDF documents
2. Extract, chunk, and embed text into **ChromaDB** (vector database)
3. On each user question:
   - Check an in-memory **query cache** first (fast path, no retrieval/LLM call)
   - On a cache miss: retrieve the top-k relevant chunks from ChromaDB
   - Build a grounded prompt from the retrieved chunks
   - Call the **Gemini API** to generate an answer
   - Save the result to the cache
4. Return the answer with **source document + chunk citations**
5. Simple evaluation metrics (latency, retrieval similarity, groundedness)
6. Interactive chat loop

Run cells top to bottom. Free to run on Colab CPU. You only need a **Gemini API key**
(get one at https://aistudio.google.com/apikey) — there is no cost for the embeddings
or vector database, only for Gemini API calls (Gemini has a generous free tier).

## Step 0: Install Dependencies

In [8]:
# Install required libraries (run once per Colab session)
# NOTE: we use the current `google-genai` SDK (the older `google-generativeai`
# package is deprecated by Google) and Gemini 3.5 Flash as the default model.
!pip install -q -U chromadb sentence-transformers PyPDF2 google-genai
print("Libraries installed successfully.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 21.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.
Libraries installed successfully.


In [9]:
import os
import time
import hashlib
import numpy as np

from google.colab import files
import PyPDF2
import chromadb
from sentence_transformers import SentenceTransformer
from google import genai
from google.genai import types

print("All imports successful!")

All imports successful!


In [10]:
%pip install -q PyPDF2 chromadb sentence-transformers google-genai


## Step 1: Set Your Gemini API Key

Get a free key from https://aistudio.google.com/apikey

The safest way in Colab is to store it as a **Colab secret** named `GEMINI_API_KEY`
(the key icon 🔑 in the left sidebar). If no secret is found, you'll be prompted to
paste it in (hidden input, not printed or saved to the notebook).

In [11]:
# Load the Gemini API key: Colab secret first, fallback to a hidden prompt
api_key = None

try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except Exception:
    api_key = None

if not api_key:
    import getpass
    api_key = getpass.getpass("Enter your Gemini API key: ")

os.environ["GEMINI_API_KEY"] = api_key

# Create the Gemini client (reads GEMINI_API_KEY from the environment)
gemini_client = genai.Client(api_key=api_key)

# Model used for answer generation. gemini-3.5-flash is the current, fast,
# free-tier-friendly model. Swap this string if Google releases a newer one.
GEMINI_MODEL = "gemini-3.5-flash"

print("Gemini client configured. Using model:", GEMINI_MODEL)

Enter your Gemini API key: ··········
Gemini client configured. Using model: gemini-3.5-flash


## Step 2: Upload PDF Documents

Select one or more PDF files from your computer (e.g. a syllabus, policy document, or handbook).

In [12]:
uploaded = files.upload()
pdf_paths = list(uploaded.keys())

print(f"Uploaded {len(pdf_paths)} file(s): {pdf_paths}")

KeyboardInterrupt: 

## Step 3: Extract and Chunk Text (with page numbers)

We extract text **page by page** (so we can cite page numbers later), then split each
page's text into overlapping word-based chunks. Chunking keeps each piece of text short
enough for the embedding model and the LLM to handle well.

In [ ]:
def extract_pages_from_pdf(path):
    """Return a list of (page_number, page_text) tuples, 1-indexed."""
    reader = PyPDF2.PdfReader(path)
    pages = []
    for i, page in enumerate(reader.pages):
        page_text = page.extract_text() or ""
        pages.append((i + 1, page_text))
    return pages


def chunk_text(text, chunk_size=400, overlap=50):
    """Split text into overlapping word-based chunks."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


# Process every uploaded PDF into chunks with source + page metadata
all_chunks = []
all_metadatas = []

for path in pdf_paths:
    pages = extract_pages_from_pdf(path)
    for page_num, page_text in pages:
        chunks = chunk_text(page_text)
        for i, c in enumerate(chunks):
            all_chunks.append(c)
            all_metadatas.append({
                "source": path,
                "page": page_num,
                "chunk_id": i
            })

print(f"Total chunks created: {len(all_chunks)}")
if all_chunks:
    print("\nSample chunk (source: {}, page {}):\n".format(
        all_metadatas[0]["source"], all_metadatas[0]["page"]))
    print(all_chunks[0][:500])
else:
    print("No chunks found. Check that your PDFs contain selectable text (not scanned images).")

## Step 4: Generate Embeddings and Store in ChromaDB

Each chunk is converted into a vector using the free local model `all-MiniLM-L6-v2`
(384-dimensional embeddings) and stored in ChromaDB along with its text and metadata
(source file + page number), so later we can cite exactly where an answer came from.

In [ ]:
# Load the embedding model (free, runs locally, downloads once)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Create an in-memory ChromaDB client and a collection.
# Swap chromadb.Client() for chromadb.PersistentClient(path="./chroma_db")
# if you want the vector database to survive across Colab sessions.
chroma_client = chromadb.Client()

# Recreate the collection cleanly if this cell is re-run
try:
    chroma_client.delete_collection(name="lgu_documents")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="lgu_documents",
    metadata={"hnsw:space": "cosine"}
)

if all_chunks:
    chunk_ids = [f"chunk_{i}" for i in range(len(all_chunks))]
    embeddings = embedding_model.encode(all_chunks, show_progress_bar=True).tolist()

    collection.add(
        ids=chunk_ids,
        documents=all_chunks,
        embeddings=embeddings,
        metadatas=all_metadatas
    )

    print(f"Stored {collection.count()} chunks in the vector database.")
    print(f"Embedding dimension: {len(embeddings[0])}")
else:
    print("No chunks to embed yet — upload PDFs in Step 2 first.")

## Step 5: Build the Retrieval Function

In [ ]:
def retrieve(query, top_k=3):
    """Embed the query and fetch the top_k most similar chunks from ChromaDB."""
    query_embedding = embedding_model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=top_k)
    retrieved_chunks = results["documents"][0]
    retrieved_metadata = results["metadatas"][0]
    retrieved_distances = results["distances"][0]
    return retrieved_chunks, retrieved_metadata, retrieved_distances


# Quick smoke test (skip if nothing has been embedded yet)
if collection.count() > 0:
    test_query = "What is this document about?"
    chunks_out, meta_out, dist_out = retrieve(test_query)
    for i, (c, m, d) in enumerate(zip(chunks_out, meta_out, dist_out)):
        print(f"\nResult {i+1} (distance: {d:.4f}, source: {m['source']}, page: {m['page']}):")
        print(c[:300])

## Step 6: Query Cache

A simple in-memory cache keyed on the normalized question text. This mirrors the
"Conversation Cache / Query Cache" box in the architecture diagram:

- **Lookup**: if we've seen this (or a near-identical) question before, return the
  saved answer instantly — no retrieval, no LLM call, no cost.
- **Save**: after a fresh answer is generated, store it in the cache.

This is a plain Python dict, so it resets when the Colab runtime restarts. If you want
it to persist across sessions, see the optional disk-persistence cell below.

In [ ]:
query_cache = {}  # normalized_query -> result dict


def normalize_query(query):
    """Normalize a query so trivially different phrasings of the same
    question (casing, extra whitespace, punctuation) hit the same cache entry."""
    q = query.strip().lower()
    q = " ".join(q.split())
    q = q.rstrip("?.! ")
    return q


def cache_key(query):
    normalized = normalize_query(query)
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


def cache_get(query):
    key = cache_key(query)
    return query_cache.get(key)


def cache_set(query, result):
    key = cache_key(query)
    query_cache[key] = result


print("Query cache ready. Current size:", len(query_cache))

In [ ]:
# OPTIONAL: persist the cache to a JSON file on disk so it survives a
# runtime restart. Run this cell once if you want that; otherwise skip it.
import json as _json

CACHE_FILE = "query_cache.json"

def save_cache_to_disk():
    with open(CACHE_FILE, "w") as f:
        _json.dump(query_cache, f)

def load_cache_from_disk():
    global query_cache
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, "r") as f:
            query_cache = _json.load(f)
        print(f"Loaded {len(query_cache)} cached entries from disk.")
    else:
        print("No cache file found yet.")

load_cache_from_disk()

## Step 7: Gemini Generation

A small wrapper around the Gemini API call, so the rest of the pipeline doesn't
need to know the SDK details.

In [ ]:
def generate_with_gemini(prompt, model=GEMINI_MODEL, max_output_tokens=512):
    response = gemini_client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            max_output_tokens=max_output_tokens,
            temperature=0.2,
        ),
    )
    return response.text or ""


## Step 8: Full RAG Pipeline (cache → retrieve → prompt → Gemini)

This function ties everything together and matches the diagram's flow:
`User question → cache lookup → (miss) retrieve from ChromaDB → build grounded
prompt → Gemini → answer with citations → save to cache`.

In [ ]:
from google.genai import errors as genai_errors
import random

def generate_with_gemini(prompt, model=GEMINI_MODEL, max_output_tokens=512,
                          max_retries=5, base_delay=2.0):
    """Call Gemini with retry + exponential backoff for transient errors
    (503 model overloaded, 429 rate limited, brief network hiccups)."""
    last_error = None
    for attempt in range(max_retries):
        try:
            response = gemini_client.models.generate_content(
                model=model,
                contents=prompt,
                config=types.GenerateContentConfig(
                    max_output_tokens=max_output_tokens,
                    temperature=0.2,
                ),
            )
            return response.text or ""
        except genai_errors.ServerError as e:
            # 503 (overloaded) / 500 (internal) — worth retrying
            last_error = e
        except genai_errors.ClientError as e:
            # 429 (rate limited) — worth retrying; anything else, fail fast
            if getattr(e, "status_code", None) == 429:
                last_error = e
            else:
                raise

        sleep_time = base_delay * (2 ** attempt) + random.uniform(0, 1)
        print(f"  Gemini busy (attempt {attempt + 1}/{max_retries}), "
              f"retrying in {sleep_time:.1f}s...")
        time.sleep(sleep_time)

    raise RuntimeError(
        f"Gemini API still unavailable after {max_retries} retries: {last_error}"
    )

## Step 9: Evaluate Performance

Simple, explainable metrics since there's no labeled gold dataset:
- **Retrieval / generation latency** — how fast each stage runs (cache hits show ~0s)
- **Average retrieval similarity** — how close retrieved chunks are to the query
- **Groundedness score** — proportion of answer words that also appear in the
  retrieved context (a simple proxy for faithfulness)
- **Answer length** — sanity check on output size

In [ ]:
from google.genai import errors as genai_errors
import random

# Tried in order. If the first model is overloaded, we fall back to the next.
FALLBACK_MODELS = ["gemini-3.5-flash", "gemini-3.1-flash-lite", "gemini-2.5-flash"]

def generate_with_gemini(prompt, models=FALLBACK_MODELS, max_output_tokens=512,
                          retries_per_model=2, base_delay=2.0):
    """Call Gemini with retry + exponential backoff per model, falling back
    to the next model in the list if one keeps returning 503/429."""
    last_error = None

    for model in models:
        for attempt in range(retries_per_model):
            try:
                response = gemini_client.models.generate_content(
                    model=model,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        max_output_tokens=max_output_tokens,
                        temperature=0.2,
                    ),
                )
                return response.text or ""
            except genai_errors.ServerError as e:
                last_error = e
            except genai_errors.ClientError as e:
                if getattr(e, "status_code", None) == 429:
                    last_error = e
                else:
                    raise

            sleep_time = base_delay * (2 ** attempt) + random.uniform(0, 1)
            print(f"  {model} busy (attempt {attempt + 1}/{retries_per_model}), "
                  f"retrying in {sleep_time:.1f}s...")
            time.sleep(sleep_time)

        print(f"  Giving up on {model}, trying next fallback model...")

    raise RuntimeError(
        f"All models unavailable after trying {models}: {last_error}"
    )

## Step 10: Interactive Chat

Ask your own questions about the uploaded documents. Type `exit` or `quit` to stop.
Repeating a question (even with different capitalization/punctuation) will be served
instantly from the cache.

In [ ]:
print("LGU-RAG Chat Assistant (type 'exit' to quit)")
print("=" * 50)

while True:
    user_query = input("\nYou: ")
    if user_query.strip().lower() in ["exit", "quit"]:
        print("Chat ended.")
        break

    if collection.count() == 0:
        print("No documents loaded yet — run Step 2-4 first.")
        continue

    result = rag_answer(user_query)
    metrics = evaluate_response(result)

    cache_tag = " (from cache)" if result["from_cache"] else ""
    print(f"\nAssistant{cache_tag}: {result['answer']}")
    print(f"\nSources: {', '.join(sorted(set(result['sources'])))}")
    print(f"(Retrieved {metrics['Chunks retrieved']} chunks | "
          f"latency {metrics['Total latency (s)']}s | "
          f"groundedness {metrics['Groundedness score (word overlap)']})")

# Optional: persist the cache so it survives a runtime restart
save_cache_to_disk()

In [1]:
%%writefile rag_pipeline.py
"""
rag_pipeline.py
Reusable RAG pipeline: PDF ingestion -> ChromaDB -> Gemini generation,
with a query cache and evaluation metrics (latency, retrieval similarity,
groundedness, and optional answer-similarity "accuracy").
"""

import os
import time
import json
import random
import hashlib

import numpy as np
import PyPDF2
import chromadb
from sentence_transformers import SentenceTransformer

from google import genai
from google.genai import types
from google.genai import errors as genai_errors


DEFAULT_FALLBACK_MODELS = ["gemini-3.5-flash", "gemini-3.1-flash-lite", "gemini-2.5-flash"]


class RAGPipeline:
    def __init__(
        self,
        gemini_api_key,
        chroma_path="./chroma_db",
        collection_name="lgu_documents",
        embed_model_name="all-MiniLM-L6-v2",
        fallback_models=None,
        cache_path="query_cache.json",
    ):
        self.gemini_client = genai.Client(api_key=gemini_api_key)
        self.fallback_models = fallback_models or DEFAULT_FALLBACK_MODELS
        self.embedding_model = SentenceTransformer(embed_model_name)

        # PersistentClient writes the vector DB to disk, so it survives a
        # kernel restart. Point chroma_path at a Google Drive folder to
        # make it survive across Colab sessions too (see setup cell).
        self.chroma_client = chromadb.PersistentClient(path=chroma_path)
        self.collection = self.chroma_client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"},
        )

        self.cache_path = cache_path
        self.cache = {}
        self.load_cache()

    # ---------- Ingestion ----------
    @staticmethod
    def _extract_pages(path):
        reader = PyPDF2.PdfReader(path)
        return [(i + 1, p.extract_text() or "") for i, p in enumerate(reader.pages)]

    @staticmethod
    def _chunk_text(text, chunk_size=400, overlap=50):
        words = text.split()
        chunks = []
        start = 0
        while start < len(words):
            end = start + chunk_size
            chunk = " ".join(words[start:end])
            if chunk.strip():
                chunks.append(chunk)
            start += chunk_size - overlap
        return chunks

    def ingest_pdfs(self, pdf_paths):
        """Extract, chunk, embed and store one or more PDF files.
        Returns a summary dict."""
        new_chunks, new_metas = [], []

        for path in pdf_paths:
            fname = os.path.basename(path)
            for page_num, page_text in self._extract_pages(path):
                for i, c in enumerate(self._chunk_text(page_text)):
                    new_chunks.append(c)
                    new_metas.append({"source": fname, "page": page_num, "chunk_id": i})

        if not new_chunks:
            return {"files": [os.path.basename(p) for p in pdf_paths], "chunks_added": 0,
                    "total_chunks": self.collection.count()}

        existing = self.collection.count()
        ids = [f"chunk_{existing + i}" for i in range(len(new_chunks))]
        embeddings = self.embedding_model.encode(new_chunks, show_progress_bar=False).tolist()

        self.collection.add(ids=ids, documents=new_chunks, embeddings=embeddings, metadatas=new_metas)

        return {
            "files": [os.path.basename(p) for p in pdf_paths],
            "chunks_added": len(new_chunks),
            "total_chunks": self.collection.count(),
        }

    # ---------- Retrieval ----------
    def retrieve(self, query, top_k=3):
        query_embedding = self.embedding_model.encode([query]).tolist()
        results = self.collection.query(query_embeddings=query_embedding, n_results=top_k)
        return results["documents"][0], results["metadatas"][0], results["distances"][0]

    # ---------- Cache ----------
    @staticmethod
    def _normalize(query):
        q = " ".join(query.strip().lower().split())
        return q.rstrip("?.! ")

    def _cache_key(self, query):
        return hashlib.sha256(self._normalize(query).encode("utf-8")).hexdigest()

    def cache_get(self, query):
        return self.cache.get(self._cache_key(query))

    def cache_set(self, query, result):
        self.cache[self._cache_key(query)] = result

    def clear_cache(self):
        self.cache = {}
        self.save_cache()

    def save_cache(self):
        with open(self.cache_path, "w") as f:
            json.dump(self.cache, f)

    def load_cache(self):
        if os.path.exists(self.cache_path):
            with open(self.cache_path, "r") as f:
                self.cache = json.load(f)

    # ---------- Generation ----------
    def _generate(self, prompt, max_output_tokens=512, retries_per_model=2, base_delay=2.0):
        last_error = None
        for model in self.fallback_models:
            for attempt in range(retries_per_model):
                try:
                    response = self.gemini_client.models.generate_content(
                        model=model,
                        contents=prompt,
                        config=types.GenerateContentConfig(
                            max_output_tokens=max_output_tokens,
                            temperature=0.2,
                        ),
                    )
                    return response.text or "", model
                except genai_errors.ServerError as e:
                    last_error = e
                except genai_errors.ClientError as e:
                    if getattr(e, "status_code", None) == 429:
                        last_error = e
                    else:
                        raise
                time.sleep(base_delay * (2 ** attempt) + random.uniform(0, 1))
        raise RuntimeError(f"All models unavailable after trying {self.fallback_models}: {last_error}")

    def _build_prompt(self, query, chunks, metas):
        context = "\n\n".join(
            f"[Source: {m['source']}, page {m['page']}]\n{c}" for c, m in zip(chunks, metas)
        )
        return (
            "You are a helpful assistant answering questions using only the context below. "
            "If the answer is not contained in the context, say \"I don't have that information "
            "in the provided documents.\" Always mention which source/page the information "
            "came from when possible.\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {query}\n\n"
            "Answer:"
        )

    # ---------- Full pipeline ----------
    def answer(self, query, top_k=3, use_cache=True):
        if use_cache:
            cached = self.cache_get(query)
            if cached is not None:
                cached = dict(cached)
                cached["from_cache"] = True
                return cached

        t0 = time.time()
        chunks, metas, distances = self.retrieve(query, top_k)
        retrieval_time = time.time() - t0

        prompt = self._build_prompt(query, chunks, metas)

        t1 = time.time()
        answer_text, model_used = self._generate(prompt)
        generation_time = time.time() - t1

        result = {
            "query": query,
            "answer": answer_text,
            "model_used": model_used,
            "context_chunks": chunks,
            "sources": [f"{m['source']} (p.{m['page']})" for m in metas],
            "retrieval_distances": distances,
            "retrieval_time": retrieval_time,
            "generation_time": generation_time,
            "from_cache": False,
        }

        if use_cache:
            self.cache_set(query, result)
            self.save_cache()

        return result

    # ---------- Evaluation ----------
    def evaluate(self, result, expected_answer=None):
        answer_words = set(result["answer"].lower().split())
        context_words = set(" ".join(result["context_chunks"]).lower().split())
        groundedness = (len(answer_words & context_words) / len(answer_words)) if answer_words else 0.0

        distances = result.get("retrieval_distances") or []
        avg_similarity = (1 - (sum(distances) / len(distances))) if distances else None

        metrics = {
            "from_cache": result.get("from_cache", False),
            "model_used": result.get("model_used", "cached"),
            "retrieval_latency_s": round(result.get("retrieval_time", 0.0), 3),
            "generation_latency_s": round(result.get("generation_time", 0.0), 3),
            "total_latency_s": round(result.get("retrieval_time", 0.0) + result.get("generation_time", 0.0), 3),
            "chunks_retrieved": len(result["context_chunks"]),
            "avg_retrieval_similarity": round(avg_similarity, 3) if avg_similarity is not None else None,
            "groundedness": round(groundedness, 3),
            "answer_length_words": len(result["answer"].split()),
        }

        # True "accuracy" needs human-labeled Q&A pairs. Without that, the
        # closest honest substitute is semantic similarity to a known-correct
        # answer, if you provide one.
        if expected_answer:
            emb = self.embedding_model.encode([result["answer"], expected_answer])
            cos_sim = float(np.dot(emb[0], emb[1]) / (np.linalg.norm(emb[0]) * np.linalg.norm(emb[1])))
            metrics["answer_similarity_to_expected"] = round(cos_sim, 3)

        return metrics

Writing rag_pipeline.py


In [2]:
import os, getpass

try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except Exception:
    api_key = None
if not api_key:
    api_key = getpass.getpass("Enter your Gemini API key: ")

# Optional: mount Drive so the vector DB + cache survive across Colab
# sessions (not just across cells in the same session):
# from google.colab import drive
# drive.mount('/content/drive')
# CHROMA_PATH = "/content/drive/MyDrive/lgu_rag/chroma_db"
# CACHE_PATH = "/content/drive/MyDrive/lgu_rag/query_cache.json"

CHROMA_PATH = "./chroma_db"
CACHE_PATH = "./query_cache.json"

from rag_pipeline import RAGPipeline
pipeline = RAGPipeline(gemini_api_key=api_key, chroma_path=CHROMA_PATH, cache_path=CACHE_PATH)

print("Pipeline ready. Documents currently in DB:", pipeline.collection.count())

Enter your Gemini API key: ··········


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Pipeline ready. Documents currently in DB: 0


In [ ]:
!pip install -q gradio
import gradio as gr
import pandas as pd

def ui_ingest(files):
    if not files:
        return "No files selected."
    paths = [f.name if hasattr(f, "name") else f for f in files]
    summary = pipeline.ingest_pdfs(paths)
    return (f"Ingested {summary['chunks_added']} new chunks from "
            f"{summary['files']}. Total chunks in DB: {summary['total_chunks']}.")

def ui_ask(query, top_k, expected_answer):
    if not query.strip():
        return "Please enter a question.", "", None
    if pipeline.collection.count() == 0:
        return "No documents in the vector DB yet — upload some PDFs first.", "", None

    result = pipeline.answer(query, top_k=int(top_k))
    metrics = pipeline.evaluate(result, expected_answer=expected_answer or None)

    sources = ", ".join(sorted(set(result["sources"])))
    metrics_df = pd.DataFrame(list(metrics.items()), columns=["Metric", "Value"])

    return result["answer"], sources, metrics_df

with gr.Blocks(title="LGU-RAG Assistant") as demo:
    gr.Markdown("# LGU-RAG Assistant\nUpload PDFs, ask questions, see evaluation metrics per query.")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 1. Upload documents")
            file_input = gr.File(label="PDF files", file_count="multiple", file_types=[".pdf"])
            ingest_btn = gr.Button("Add to Vector DB")
            ingest_status = gr.Textbox(label="Ingestion status", interactive=False)

        with gr.Column(scale=2):
            gr.Markdown("### 2. Ask a question")
            query_box = gr.Textbox(label="Your question", placeholder="What is this document about?")
            top_k_slider = gr.Slider(1, 10, value=3, step=1, label="Chunks to retrieve (top_k)")
            expected_box = gr.Textbox(label="Expected answer (optional, for accuracy check)")
            ask_btn = gr.Button("Ask", variant="primary")

            answer_box = gr.Textbox(label="Answer", lines=5, interactive=False)
            sources_box = gr.Textbox(label="Sources", interactive=False)
            metrics_table = gr.Dataframe(label="Evaluation metrics", interactive=False)

    ingest_btn.click(ui_ingest, inputs=[file_input], outputs=[ingest_status])
    ask_btn.click(ui_ask, inputs=[query_box, top_k_slider, expected_box],
                  outputs=[answer_box, sources_box, metrics_table])

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://be21b96f2ffecbdee5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
